# DenseNet121 — Cross-Crop Knowledge Distillation from Lentil Teacher (Phase 3)

**Objective:** Train a DenseNet121 student on the Beans dataset using feature-based knowledge distillation from a ResNet101 teacher model that was trained on the Lentil dataset.

**Teacher:** ResNet101 (trained on Lentil disease classification, 4 classes — frozen during distillation)
**Student:** DenseNet121 (trained on Beans disease classification, 3 classes)

**KD Method:** Feature-based (hint-based) distillation
- Teacher hint layers: `conv4_block23_out` (14x14x1024) and `conv5_block3_out` (7x7x2048)
- Student hint layers: `conv4_block24_concat` (14x14x1024) and `conv5_block16_concat` (7x7x1024)
- Student features are projected via 1x1 Conv2D to match teacher channel dimensions where needed
- Loss: `total_loss = classification_loss + 0.1 * normalized_feature_MSE`

**Scientific Rationale:** Both lentils and beans belong to the Fabaceae family, sharing similar leaf morphology and disease manifestations (e.g., rust). This taxonomical similarity enables meaningful cross-crop knowledge transfer. DenseNet121's dense connectivity pattern promotes feature reuse, making it efficient for learning from teacher features.

**Context:** This is Phase 3 of the project. Results are compared against the baseline (Phase 2, without KD) to quantify the benefit of cross-crop knowledge distillation.

In [1]:
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D

## Step 1: Load and Freeze the Teacher Model

Loading the ResNet101 teacher model trained on the Lentil Augmented Dataset. All teacher parameters are frozen — the teacher serves only as a feature extractor during distillation.

In [2]:
teacher = tf.keras.models.load_model(
    r"C:\Users\k nithin\Downloads\M.Tech Project\Code\ResNet101_lentil_10.keras",
    compile=False
)

teacher.trainable = False  # 🔒 Freeze teacher completely
print("Teacher loaded and frozen")


Teacher loaded and frozen


In [3]:
print("Trainable variables in teacher:",
      len(teacher.trainable_variables))

Trainable variables in teacher: 0


## Step 2: Define Teacher Hint Layers

Selecting intermediate layers from the teacher whose feature representations will be transferred to the student. These layers capture progressively abstract visual features relevant to disease patterns.

In [4]:
TEACHER_HINT_LAYERS = [
    "conv4_block23_out",  # deep semantic features
    "conv5_block3_out"    # highest-level features
]

In [5]:
teacher_feature_extractor = Model(
    inputs=teacher.input,
    outputs=[teacher.get_layer(name).output for name in TEACHER_HINT_LAYERS],
    name="teacher_feature_extractor"
)

teacher_feature_extractor.trainable = False

In [6]:
teacher_feature_extractor.summary()

Model: "teacher_feature_extractor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_pad (ZeroPadding2D)     │ (None, 230, 230, 3)       │               0 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_conv (Conv2D)           │ (None, 112, 112, 64)      │           9,472 │ conv1_pad[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_bn (BatchNormalization) │ (None, 112, 112, 64)      │             256 │ conv1_conv[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_relu (Activation)       │ (None, 112, 112, 64)      │               0 │ conv1_bn[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ pool1_pad (ZeroPadding2D)     │ (None, 114, 114, 64)      │               0 │ conv1_relu[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ pool1_pool (MaxPooling2D)     │ (None, 56, 56, 64)        │               0 │ pool1_pad[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_conv (Conv2D)  │ (None, 56, 56, 64)        │           4,160 │ pool1_pool[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_bn             │ (None, 56, 56, 64)        │             256 │ conv2_block1_1_conv[0][0]  │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_relu           │ (None, 56, 56, 64)        │               0 │ conv2_block1_1_bn[0][0]    │
│ (Activation)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_conv (Conv2D)  │ (None, 56, 56, 64)        │          36,928 │ conv2_block1_1_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_bn             │ (None, 56, 56, 64)        │             256 │ conv2_block1_2_conv[0][0]  │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_relu           │ (None, 56, 56, 64)        │               0 │ conv2_block1_2_bn[0][0]    │
│ (Activation)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_0_conv (Conv2D)  │ (None, 56, 56, 256)       │          16,640 │ pool1_pool[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_3_conv (Conv2D)  │ (None, 56, 56, 256)       │          16,640 │ conv2_block1_2_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼───────────────

 Total params: 42,658,176 (162.73 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 42,658,176 (162.73 MB)

## Step 3: Build and Configure the Student Model

Building the DenseNet121 student model with ImageNet-pretrained weights. Student hint layers (`conv4_block24_concat` at 14x14x1024 and `conv5_block16_concat` at 7x7x1024) are selected to correspond to similar depth/resolution as the teacher hint layers. DenseNet's dense block concatenation results in different channel growth compared to ResNet.

In [7]:
student_base = DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

In [8]:
STUDENT_HINT_LAYERS = [
    "conv4_block24_concat",   # mid-deep semantic
    "conv5_block16_concat"    # final semantic
]

In [9]:
student_feature_extractor = tf.keras.Model(
    inputs=student_base.input,
    outputs=[
        student_base.get_layer(name).output
        for name in STUDENT_HINT_LAYERS
    ]
)

## Step 4: Feature Projection Layers

Adding 1x1 convolutional projection layers to align the student's feature map channel dimensions with the teacher's. For DenseNet121, the first hint layer matches the teacher (1024 channels), but the second hint layer (1024 channels) needs projection to match the teacher's 2048 channels.

In [10]:
# Get teacher feature shapes
teacher_shapes = [
    teacher_feature_extractor.output[0].shape[-1],
    teacher_feature_extractor.output[1].shape[-1],
]

# Projection layers to match teacher channels
proj_layers = [
    tf.keras.layers.Conv2D(teacher_shapes[0], kernel_size=1, padding="same", name="proj_0"),
    tf.keras.layers.Conv2D(teacher_shapes[1], kernel_size=1, padding="same", name="proj_1"),
]


In [11]:
def projection_layer(x, out_channels, name):
    return tf.keras.layers.Conv2D(
        out_channels,
        kernel_size=1,
        padding="same",
        name=name
    )(x)

In [12]:
teacher_channels = [
    teacher_feature_extractor.output[i].shape[-1]
    for i in range(len(teacher_feature_extractor.output))
]

print("Teacher channels:", teacher_channels)

Teacher channels: [1024, 2048]


In [13]:
student_features_raw = student_feature_extractor.output

In [14]:
projected_student_features = []

for i, s_feat in enumerate(student_features_raw):

    student_channels = s_feat.shape[-1]
    teacher_ch = teacher_channels[i]

    if student_channels != teacher_ch:
        # Only project if channels mismatch
        s_feat = tf.keras.layers.Conv2D(
            teacher_ch,
            kernel_size=1,
            padding="same",
            name=f"proj_{i}"
        )(s_feat)

    projected_student_features.append(s_feat)

In [15]:
import numpy as np

dummy_x = np.random.rand(2, 224, 224, 3).astype("float32")

teacher_feats = teacher_feature_extractor(dummy_x)
student_feats = student_feature_extractor(dummy_x)

for i, (tf, sf) in enumerate(zip(teacher_feats, student_feats)):
    print(f"\nLayer {i}")
    print("Teacher shape:", tf.shape)
    print("Student shape:", sf.shape)


Layer 0
Teacher shape: (2, 14, 14, 1024)
Student shape: (2, 14, 14, 1024)

Layer 1
Teacher shape: (2, 7, 7, 2048)
Student shape: (2, 7, 7, 1024)


In [16]:
for i, (tf, sf) in enumerate(zip(teacher_feats, student_feats)):
    print(f"\nLayer {i}")
    print("Teacher mean/std:", tf.numpy().mean(), tf.numpy().std())
    print("Student mean/std:", sf.numpy().mean(), sf.numpy().std())


Layer 0
Teacher mean/std: 1.5218683 2.207982
Student mean/std: -0.034180872 0.114204034

Layer 1
Teacher mean/std: 0.08983443 0.69783646
Student mean/std: -0.0059493617 0.05693461


## Step 5: Knowledge Distillation Loss

Defining the feature-based KD loss function:
1. **Per-channel normalization** of both teacher and student feature maps (zero mean, unit variance)
2. **Mean Squared Error (MSE)** between normalized feature maps
3. **Combined loss:** `total = classification_loss + 0.1 * feature_MSE` where lambda = 0.1

In [17]:
def normalize_feature(f):
    # Normalize per-channel
    mean = tf.reduce_mean(f, axis=[1, 2], keepdims=True)
    std = tf.math.reduce_std(f, axis=[1, 2], keepdims=True) + 1e-6
    return (f - mean) / std

In [18]:
def feature_kd_loss(teacher_feats, student_feats):
    loss = 0.0
    for t_feat, s_feat in zip(teacher_feats, student_feats):
        t_feat = tf.stop_gradient(normalize_feature(t_feat))
        s_feat = normalize_feature(s_feat)
        loss += tf.reduce_mean(tf.square(t_feat - s_feat))
    return loss

## Step 6: Student Classification Head

Adding the final classification layers to the student model with Dropout (0.7) for regularization. The student outputs both class logits (for classification loss) and projected intermediate features (for KD loss).

In [19]:
NUM_CLASSES = 3  # example: healthy, rust, anthracnose

In [20]:
import tensorflow as tf
x = student_base.output
x = GlobalAveragePooling2D()(x)
x = Dense(8, activation="relu")(x)
x = tf.keras.layers.Dropout(0.7)(x)
student_logits = Dense(NUM_CLASSES, activation=None, name="bean_logits")(x)

In [21]:
# Get original student feature outputs
student_features_raw = student_feature_extractor.output

# Apply projections
student_features_projected = [
    proj_layers[i](student_features_raw[i])
    for i in range(len(student_features_raw))
]

# Build full student model
student_model = Model(
    inputs=student_base.input,
    outputs={
        "logits": student_logits,
        "features": student_features_projected
    }
)


In [22]:
import tensorflow as tf

In [23]:
classification_loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True
)

In [24]:
KD_WEIGHT = 0.1 # λ

In [25]:
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)

## Step 7: Custom Training Step

Implementing a custom training step using `tf.GradientTape` that:
1. Passes input through the frozen teacher to get target feature maps
2. Passes input through the student to get predicted features and class logits
3. Computes both classification loss and feature-matching KD loss
4. Updates only the student's parameters via backpropagation

In [26]:
#@tf.function
def train_step(images, labels):
    # 1. Teacher forward (features only)
    teacher_feats = teacher_feature_extractor(images)

    with tf.GradientTape() as tape:
        # 2. Student forward
        outputs = student_model(images, training=True)
        student_logits = outputs["logits"]
        student_feats = outputs["features"]

        # 3. Losses
        cls_loss = classification_loss_fn(labels, student_logits)
        kd_loss = feature_kd_loss(teacher_feats, student_feats)

        total_loss = cls_loss + KD_WEIGHT * kd_loss

    # 4. Backprop (student only)
    grads = tape.gradient(total_loss, student_model.trainable_variables)
    optimizer.apply_gradients(zip(grads, student_model.trainable_variables))

    return {
        "total_loss": total_loss,
        "cls_loss": cls_loss,
        "kd_loss": kd_loss
    }

In [27]:
dummy_y = tf.constant([0, 1])  # fake labels

losses = train_step(dummy_x, dummy_y)
print(losses)

{'total_loss': <tf.Tensor: shape=(), dtype=float32, numpy=0.9253864288330078>, 'cls_loss': <tf.Tensor: shape=(), dtype=float32, numpy=0.6004846096038818>, 'kd_loss': <tf.Tensor: shape=(), dtype=float32, numpy=3.2490181922912598>}


## Step 8: Load Beans Dataset

Loading the Beans Leaf Disease Dataset (3 classes: Anthracnose, Healthy, Rust) with train/validation/test splits.

In [28]:
import tensorflow as tf
import pathlib

DATA_DIR = pathlib.Path(r"C:\Users\k nithin\Downloads\M.Tech Project\Beans dataset")
IMG_SIZE = (224,224)
BATCH_SIZE = 8
SEED = 42
NUM_CLASSES = 3


In [29]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    labels="inferred",
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
    validation_split=0.3,
    subset="training"
)

Found 59069 files belonging to 3 classes.
Using 41349 files for training.


In [30]:
val_test_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    labels="inferred",
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
    validation_split=0.3,
    subset="validation"
)

Found 59069 files belonging to 3 classes.
Using 17720 files for validation.


In [31]:
val_batches = tf.data.experimental.cardinality(val_test_ds).numpy()
test_ds = val_test_ds.take(val_batches // 2)
val_ds = val_test_ds.skip(val_batches // 2)

In [32]:
class_names = train_ds.class_names
print("Class order:", class_names)

Class order: ['anthra', 'healthy', 'rust']


In [33]:
train_batches = tf.data.experimental.cardinality(train_ds).numpy()
print("Train batches per epoch:", train_batches)
print("Approx images per epoch:", train_batches * BATCH_SIZE)

Train batches per epoch: 5169
Approx images per epoch: 41352


In [34]:
from tensorflow.keras.applications.resnet import preprocess_input

def preprocess(image, label):
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)
    return image, label

In [35]:
train_ds = train_ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
test_ds = test_ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)

In [36]:
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)

In [37]:
import time

images, labels = next(iter(train_ds))

start = time.time()
losses = train_step(images, labels)
end = time.time()

print("One train_step time (seconds):", end - start)
print(losses)

One train_step time (seconds): 5.133049249649048
{'total_loss': <tf.Tensor: shape=(), dtype=float32, numpy=1.4554728269577026>, 'cls_loss': <tf.Tensor: shape=(), dtype=float32, numpy=1.1120929718017578>, 'kd_loss': <tf.Tensor: shape=(), dtype=float32, numpy=3.43379807472229>}


## Step 9: Training Loop

Training the student model with the combined classification + KD loss. The teacher remains frozen throughout — only student weights are updated.

In [38]:
STEPS_PER_EPOCH = 40
EPOCHS = 10

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    # ---------- TRAINING ----------
    step = 0
    for images, labels in train_ds:
        losses = train_step(images, labels)
        step += 1

        if step >= STEPS_PER_EPOCH:
            break

    print(
        f"Train | total: {losses['total_loss']:.4f} | "
        f"cls: {losses['cls_loss']:.4f} | "
        f"kd: {losses['kd_loss']:.4f}"
    )

    # ---------- VALIDATION ----------
    # ✅ ADD THESE TWO LINES
    val_cls_loss = tf.keras.metrics.Mean()
    val_acc = tf.keras.metrics.SparseCategoricalAccuracy()

    for images, labels in val_ds:
        outputs = student_model(images, training=False)
        logits = outputs["logits"]

        loss = classification_loss_fn(labels, logits)
        val_cls_loss.update_state(loss)
        val_acc.update_state(labels, logits)

    print(
        f"Val   | cls_loss: {val_cls_loss.result():.4f} | "
        f"acc: {val_acc.result():.4f}"
    )



Epoch 1/10
Train | total: 1.5626 | cls: 1.2268 | kd: 3.3583
Val   | cls_loss: 0.7432 | acc: 0.6593

Epoch 2/10
Train | total: 1.2064 | cls: 0.8923 | kd: 3.1409
Val   | cls_loss: 0.4654 | acc: 0.7616

Epoch 3/10
Train | total: 0.7366 | cls: 0.4431 | kd: 2.9352
Val   | cls_loss: 0.2039 | acc: 0.9436

Epoch 4/10
Train | total: 0.8563 | cls: 0.5719 | kd: 2.8442
Val   | cls_loss: 0.1440 | acc: 0.9771

Epoch 5/10
Train | total: 0.8414 | cls: 0.5631 | kd: 2.7829
Val   | cls_loss: 0.1144 | acc: 0.9888

Epoch 6/10
Train | total: 0.4808 | cls: 0.2027 | kd: 2.7813
Val   | cls_loss: 0.1027 | acc: 0.9886

Epoch 7/10
Train | total: 0.5916 | cls: 0.3230 | kd: 2.6856
Val   | cls_loss: 0.0973 | acc: 0.9879

Epoch 8/10
Train | total: 0.8267 | cls: 0.5624 | kd: 2.6431
Val   | cls_loss: 0.1105 | acc: 0.9827

Epoch 9/10
Train | total: 0.7289 | cls: 0.4671 | kd: 2.6175
Val   | cls_loss: 0.0759 | acc: 0.9895

Epoch 10/10
Train | total: 0.5875 | cls: 0.3405 | kd: 2.4692
Val   | cls_loss: 0.0659 | acc: 0.9919